# NEXUS Training Notebook — Google Colab

**Run all cells top-to-bottom.**

| Cell | What it does |
|---|---|
| 1 | Clone / pull the repo, set env vars |
| 2 | Install dependencies + download ATTNSOM dataset |
| 3 | Environment diagnostic (GPU / RAM) |
| 4 | Single-molecule smoke test (fast forward pass) |
| 5 | Training — reads `scripts/colab_train.py` (always up-to-date after git pull) |

All memory-sensitive parameters (quantum grid resolution, chunk size, dynamics mode)
are controlled by the GPU profile auto-detected in `scripts/colab_train.py`.

**GPU profile auto-selection:**
| GPU VRAM | Profile selected | Physics mode |
|---|---|---|
| ≥ 70 GB (A100-80, H100) | `ultra_vram` | full Clifford dynamics |
| 35–70 GB (A100-40, L40S) | `high_vram` | lite dynamics |
| < 35 GB (T4, V100-16) | `standard` | lite dynamics |

Override: set `NEXUS_COLAB_GPU_PROFILE=standard|high_vram|ultra_vram` in Cell 1.

In [ ]:
import os, subprocess

REPO     = 'https://github.com/Nikku03/enzyme_Software.git'
REPO_DIR = '/content/enzyme_Software'

# ── set allocator config BEFORE torch is imported so the CUDA caching
# ── allocator picks it up on first use.  expandable_segments lets PyTorch
# ── reuse fragmented reserved-but-unallocated blocks (saves ~8 GB on A100).
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# ── GPU profile — leave as 'auto' to let colab_train.py pick the right
# ── tier for your GPU, or override with one of: standard | high_vram | ultra_vram
# os.environ['NEXUS_COLAB_GPU_PROFILE'] = 'ultra_vram'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
        capture_output=True, text=True,
    )
    print(result.stdout.strip() or 'already up to date')
    if result.returncode != 0:
        print('git pull stderr:', result.stderr)

os.chdir(REPO_DIR)
print('repo :', os.getcwd())
print('PYTORCH_ALLOC_CONF :', os.environ.get('PYTORCH_ALLOC_CONF', '(not set)'))
profile_override = os.environ.get('NEXUS_COLAB_GPU_PROFILE', 'auto')
print('gpu_profile override:', profile_override, '(auto = detect from VRAM)')

In [ ]:
!bash scripts/setup_colab_nexus.sh /content/enzyme_Software

In [ ]:
# Smoke test — single forward pass to verify the model loads correctly.
# Output is captured and printed; a failure warns but does NOT block Cell 5.
import subprocess, os, sys

result = subprocess.run(
    [
        sys.executable,
        'scripts/colab_smoke_test.py',
        '--sdf', 'data/ATTNSOM/cyp_dataset/3A4.sdf',
        '--gpu-profile', os.environ.get('NEXUS_COLAB_GPU_PROFILE', 'ultra_vram'),
        '--sample-index', '-1',
        '--steps', '1',
        '--forward-only',
    ],
    cwd='/content/enzyme_Software',
    env={**os.environ},
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print('--- stderr ---')
    print(result.stderr[-4000:])  # last 4 KB of stderr
if result.returncode != 0:
    print(f'\n⚠ Smoke test exited with code {result.returncode}.')
    print('Check the output above, then decide whether to proceed to Cell 5.')
else:
    print('✓ Smoke test passed.')

In [ ]:
!export PYTORCH_ALLOC_CONF=expandable_segments:True && \
python scripts/colab_smoke_test.py \
  --sdf data/ATTNSOM/cyp_dataset/3A4.sdf \
  --gpu-profile ultra_vram \
  --sample-index -1 \
  --steps 1 \
  --forward-only

In [ ]:
# Training code lives in scripts/colab_train.py so git pull always updates it.
exec(open("/content/enzyme_Software/scripts/colab_train.py").read())